# Exercise 6.2: Top-pair production with Pythia

From *Programming in High Energy Particle Physics*, Chapter 6

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch06/ex06_2.ipynb)

Exercise 6.2 Top-pair production with Pythia Use Pythia to generate \(t\bar{t}\) events at \(\sqrt{s} = 13\) TeV. Plot the top quark \(p_T\) distribution, the \(t\bar{t}\) invariant mass, and the jet multiplicity. Compare with theoretical expectations.

<details><summary>Hint</summary>

Start from Listing 6.5 with Top:gg2ttbar = on and Top:qqbar2ttbar = on in place of the Z settings. Find the tops as the last copies of particles with abs(id()) == 6 ( event[i].iBotCopyId() helps) before they decay. For jets, cluster the final-state particles with FastJet (anti-\(k_T\), \(R = 0.4\), see Chapter 8 ) or with Pythia's built-in SlowJet . The \(m_{t\bar{t}}\) distribution starts at \(2m_t \approx 345\) GeV.

</details>

### Colab setup

The first run takes about 3–5 minutes and restarts the runtime once. After restart run all cells from the top. Pythia 8 is installed from conda-forge; this notebook compiles a small C++ program following Listing 6.5, then plots its CSV output in Python.

In [ ]:
import sys, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and importlib.util.find_spec("pythia8") is None:
    %pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
import importlib.util
if IN_COLAB and importlib.util.find_spec("pythia8") is None:
    !mamba install -q -y -c conda-forge pythia8
if IN_COLAB:
    %pip install -q numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt

### Generate top pairs and cluster final-state jets

`SlowJet` counts inclusive jets from final-state particles, including top-decay products. The 25 GeV threshold and acceptance affect the multiplicity.

In [ ]:
%%writefile ex06_2.cc
#include "Pythia8/Pythia.h"
#include <fstream>
#include <iostream>
int main() {
  Pythia8::Pythia pythia;
  pythia.readString("Beams:eCM = 13000.");
  pythia.readString("Top:gg2ttbar = on");
  pythia.readString("Top:qqbar2ttbar = on");
  pythia.readString("Random:setSeed = on");
  pythia.readString("Random:seed = 42");
  if(!pythia.init()) return 1;
  Pythia8::SlowJet jetFinder(-1,0.4,25.,2.5);
  std::ofstream out("ex06_2_events.csv");
  out << "top_pt,antitop_pt,tt_mass,n_jets\n";
  const int nEvents=1500;
  for(int ev=0;ev<nEvents;++ev) {
    if(!pythia.next()) continue;
    // TODO: locate last copies of top and antitop via iBotCopyId().
    // TODO: run jetFinder.analyze(pythia.event), count jets, and write one CSV row.
  }
  pythia.stat();
}


In [ ]:
!c++ -std=c++17 -O2 ex06_2.cc -o ex06_2 $(pythia8-config --cxxflags --libs)
!./ex06_2

In [ ]:
import pathlib
p=pathlib.Path("ex06_2_events.csv")
assert p.exists()
print("Self-check: after completing the loop, the CSV should contain many rows and m_tt should begin near 345 GeV.")

### Plot the observables

In [ ]:
# TODO: read the CSV with np.genfromtxt and plot top pT, m_tt, and jet multiplicity with units.